# CIFAR10 CNN PyTorch complete

In [3]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [4]:
print(torch.__version__)

2.11.0+cpu


In [5]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cpu


In [13]:
class_names = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
]

In [6]:
transet=torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)

100%|██████████| 170M/170M [16:28<00:00, 173kB/s]


In [7]:
testset=torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=True, transform=transform)

In [8]:
batch=32
tran_loader=torch.utils.data.DataLoader(transet, batch_size=batch,
                                          shuffle=True, num_workers=2)
test_loader=torch.utils.data.DataLoader(testset, batch_size=batch,
                                          shuffle=False, num_workers=2)

In [19]:
class CNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(3, 6, 5)
    self.conv2 = nn.Conv2d(6, 16, 5)
    self.conv3 = nn.Conv2d(16, 120, 5)

    self.maxpool = nn.MaxPool2d(2, 2)
    self.rulu = nn.ReLU()

    self.fc1 = nn.Linear(120, 84)
    self.fc2 = nn.Linear(84, 10)

  def forward(self, x):

    x = self.rulu(self.conv1(x))
    x = self.maxpool(x)
    x = self.rulu(self.conv2(x))
    x = self.maxpool(x)
    x = self.rulu(self.conv3(x))
    x = x.view(x.size(0), -1)
    x = self.rulu(self.fc1(x))
    x = self.fc2(x)
    return x

In [23]:
model = CNN().to(device)

print(model)

CNN(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (conv3): Conv2d(16, 120, kernel_size=(5, 5), stride=(1, 1))
  (maxpool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (rulu): ReLU()
  (fc1): Linear(in_features=120, out_features=84, bias=True)
  (fc2): Linear(in_features=84, out_features=10, bias=True)
)


In [24]:
sample_images, sample_labels = next(iter(tran_loader))

sample_images = sample_images.to(device)

outputs = model(sample_images)

print("Input shape:", sample_images.shape)
print("Output shape:", outputs.shape)

Input shape: torch.Size([32, 3, 32, 32])
Output shape: torch.Size([32, 10])


In [25]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [27]:
num_epochs = 10

train_losses = []
train_accuracies = []

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0
    correct_predictions = 0
    total_images = 0

    for images, labels in tran_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct_predictions += (
            predictions == labels
        ).sum().item()

        total_images += labels.size(0)

    epoch_loss = running_loss / len(tran_loader)

    epoch_accuracy = (
        correct_predictions / total_images
    ) * 100

    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {epoch_accuracy:.2f}%"
    )

Epoch [1/10] Loss: 1.5922 Accuracy: 41.65%
Epoch [2/10] Loss: 1.2913 Accuracy: 53.55%
Epoch [3/10] Loss: 1.1584 Accuracy: 58.73%
Epoch [4/10] Loss: 1.0723 Accuracy: 61.77%
Epoch [5/10] Loss: 1.0058 Accuracy: 64.32%
Epoch [6/10] Loss: 0.9552 Accuracy: 65.81%
Epoch [7/10] Loss: 0.9087 Accuracy: 67.80%
Epoch [8/10] Loss: 0.8665 Accuracy: 69.13%
Epoch [9/10] Loss: 0.8317 Accuracy: 70.34%
Epoch [10/10] Loss: 0.8005 Accuracy: 71.30%
